<a href="https://colab.research.google.com/github/YuriArduino/TTS-Text-to-Speech-Audio-Synthesis-Lab/blob/main/experimento_1_entrevista/pipeline_entrevista.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Teste em simular uma entrevista real com duas vozes, foi realizado extração, limpeza, tratamento de texto enfim convertido para voz. O notebook documenta o processo de desenvolvimento e os desafios encontrados.

In [ ]:
pip install pypdf pyttsx3

In [ ]:
# Install eSpeak-ng, which is required by pyttsx3 for speech synthesis
!apt-get update && apt-get install -y espeak-ng

In [ ]:
!pip install pypdf gTTS

In [ ]:
import pypdf
import pyttsx3

# 1. Initialize the Text-to-Speech engine
engine = pyttsx3.init()

# Optional: Adjust voice speed (default is usually around 200)
engine.setProperty('rate', 175)

# 2. Open and read the PDF file
pdf_path = "/content/v40n73a02.pdf"  # Replace with your PDF path

with open(pdf_path, "rb") as file:
    reader = pypdf.PdfReader(file)

    # Loop through all pages and read them aloud
    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()

        if text.strip():  # Skip empty pages
            print(f"Reading page {page_num + 1}...")
            engine.say(text)
            engine.runAndWait()

In [ ]:
import pypdf
from gtts import gTTS
from IPython.display import Audio, display

# 1. Extrair texto do PDF
pdf_path = "/content/v40n73a02.pdf"  # Mude para o nome do seu arquivo enviado no Colab
full_text = ""

print("Lendo o PDF...")
reader = pypdf.PdfReader(pdf_path)

# Vamos ler apenas as primeiras páginas primeiro para testar rápido
for page in reader.pages[:3]:  # Limitado a 3 páginas para teste rápido
    text = page.extract_text()
    if text:
        full_text += text + "\n"

# 2. Converter para áudio e dar Play no navegador
if full_text.strip():
    print("Gerando voz humana (gTTS)...")
    # lang='pt' para português ou lang='en' para inglês
    tts = gTTS(text=full_text, lang='pt', slow=False)

    # Salva o arquivo temporário no Colab
    audio_file = "audio_teste.mp3"
    tts.save(audio_file)

    print("Pronto! Clique no botão de Play abaixo para ouvir:")
    # Cria o player de áudio direto na tela do Colab
    display(Audio(audio_file, autoplay=False))
else:
    print("Erro: Nenhum texto foi encontrado no PDF. O arquivo pode ser uma imagem/escaneado.")


In [ ]:
!pip install pypdf edge-tts

In [ ]:
import pypdf
import asyncio
import edge_tts
from IPython.display import Audio, display

# 1. Extrair texto do PDF
pdf_path = "/content/v40n73a02.pdf"  # Mude para o nome do seu arquivo
full_text = ""

print("Lendo o PDF...")
reader = pypdf.PdfReader(pdf_path)

# Testando com as 3 primeiras páginas para ser rápido
for page in reader.pages[:3]:
    text = page.extract_text()
    if text:
        full_text += text + "\n"

# 2. Função assíncrona para gerar a voz realista
async def gerar_voz_realista(texto, arquivo_saida):
    # Vozes PT-BR excelentes: 'pt-BR-FranciscaNeural' (Feminina) ou 'pt-BR-AntonioNeural' (Masculina)
    voice = "pt-BR-FranciscaNeural"

    communicate = edge_tts.Communicate(texto, voice)
    await communicate.save(arquivo_saida)

# 3. Executar e tocar no Colab
if full_text.strip():
    print("Gerando voz realista de IA... Por favor, aguarde.")
    audio_file = "audio_realista.mp3"

    # Executa a função assíncrona dentro do Colab
    await gerar_voz_realista(full_text, audio_file)

    print("Pronto! Ouça a nova voz no player abaixo:")
    display(Audio(audio_file, autoplay=False))
else:
    print("Erro: Nenhum texto encontrado no PDF.")


In [ ]:
import pypdf
import asyncio
import edge_tts
import re
from IPython.display import Audio, display

def limpar_texto_pdf(texto_puro):
    """
    Remove ruídos comuns de PDFs como quebras de linha erradas,
    hifens de separação silábica e normaliza pontuações.
    """
    if not texto_puro:
        return ""

    # 1. Junta palavras separadas por hífen no final da linha (ex: com- \n putador -> computador)
    texto = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', texto_puro)

    # 2. Corrige quebras de linha no meio de frases, mantendo apenas parágrafos reais
    # Substitui quebra de linha simples por espaço, mas preserva quebras duplas (\n\n)
    texto = re.sub(r'(?<!\n)\n(?!\n)', ' ', texto)

    # 3. Limpa travessões problemáticos que quebram o ritmo (ajusta espaçamento)
    texto = re.sub(r'—\s*', '— ', texto)  # Garante espaço após travessão longo
    texto = re.sub(r'-\s*', '- ', texto)   # Garante espaço após hífen/travessão curto

    # 4. Remove múltiplos espaços em branco gerados pela limpeza
    texto = re.sub(r'[ \t]+', ' ', texto)

    return texto.strip()

# 1. Extrair e Tratar o texto do PDF
pdf_path = "/content/v40n73a02.pdf"  # Substitua pelo nome do seu arquivo
texto_bruto = ""

print("Lendo e tratando o texto do PDF...")
reader = pypdf.PdfReader(pdf_path)

# Testando com as 3 primeiras páginas
for page in reader.pages[:3]:
    text = page.extract_text()
    if text:
        texto_bruto += text + "\n"

# Aplica o tratamento inteligente
texto_tratado = limpar_texto_pdf(texto_bruto)

# 2. Função assíncrona para gerar a voz realista
async def gerar_voz_realista(texto, arquivo_saida):
    # 'pt-BR-FranciscaNeural' ou 'pt-BR-AntonioNeural'
    voice = "pt-BR-FranciscaNeural"
    communicate = edge_tts.Communicate(texto, voice)
    await communicate.save(arquivo_saida)

# 3. Executar e tocar no Colab
if texto_tratado:
    print("Gerando áudio com cadência corrigida... Aguarde.")
    audio_file = "audio_tratado.mp3"

    await gerar_voz_realista(texto_tratado, audio_file)

    print("Pronto! Ouça a versão com ritmo corrigido:")
    display(Audio(audio_file, autoplay=False))
else:
    print("Erro: Nenhum texto extraído.")


In [ ]:
!pip install --force-reinstall pypdf edge-tts pydub numpy

In [ ]:
import pypdf
import asyncio
import edge_tts
import re
import io
import numpy as np
from pydub import AudioSegment
# CORREÇÃO 1: O nome correto da função é detect_nonsilent
from pydub.silence import detect_nonsilent
from IPython.display import Audio, display
from dataclasses import dataclass

# --- SUA ESTRUTURA DE MÉTRICAS ADAPTADA PARA TTS (AUDIO PROCESSOR) ---
@dataclass
class AudioProcessorConfig:
    # Configurações de Voz
    voice_name: str = "pt-BR-FranciscaNeural"

    # Configurações para Energy-Based End Trimming & Padding (Métricas de Silêncio)
    enable_energy_trimming: bool = True
    trim_energy_threshold_db: int = -40  # Limiar de energia em dB (equivalente ao RMS threshold)
    trim_silence_duration_ms: int = 150  # Duração mínima para considerar silêncio

    # Ajuste de Cadência e Respiração (VAD reverso para subdivisão de segmentos)
    optimize_max_segment_duration_s: int = 20
    pause_between_paragraphs_ms: int = 800  # Tempo de respiração entre blocos de texto
    vad_fallback_min_pause_s: float = 0.2

# --- FILTRO DE TEXTO ANTERIOR ---
def limpar_texto_pdf(texto_puro):
    if not texto_puro: return ""
    texto = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', texto_puro)
    texto = re.sub(r'(?<!\n)\n(?!\n)', ' ', texto)
    texto = re.sub(r'—\s*', '— ', texto)
    texto = re.sub(r'-\s*', '- ', texto)
    texto = re.sub(r'[ \t]+', ' ', texto)
    return texto.strip()

# --- PROCESSADOR DE ÁUDIO AVANÇADO (ESTILO PYANNOTE/WHISPER VAD) ---
class AdvancedAudioProcessor:
    def __init__(self, config: AudioProcessorConfig):
        self.config = config

    def process_segment(self, audio_segment: AudioSegment) -> AudioSegment:
        """
        Aplica trimming baseado em energia e normalização no segmento gerado
        usando detect_nonsilent e adicionando padding.
        """
        if not self.config.enable_energy_trimming:
            return audio_segment

        if not audio_segment:
            return AudioSegment.empty()

        # CORREÇÃO 2: Utilizando detect_nonsilent com o nome correto
        non_silent_chunks = detect_nonsilent(
            audio_segment,
            min_silence_len=self.config.trim_silence_duration_ms,
            silence_thresh=self.config.trim_energy_threshold_db
        )

        if not non_silent_chunks:
            return AudioSegment.empty()

        # Get the start of the first non-silent chunk and the end of the last non-silent chunk
        start_trim_ms = non_silent_chunks[0][0]
        end_trim_ms = non_silent_chunks[-1][1]

        # Apply padding by extending the start and end by trim_silence_duration_ms
        start_ms = max(0, start_trim_ms - self.config.trim_silence_duration_ms)
        end_ms = min(len(audio_segment), end_trim_ms + self.config.trim_silence_duration_ms)

        trimmed_audio = audio_segment[start_ms:end_ms]
        return trimmed_audio

    async def text_to_speech_pipeline(self, texto_completo: str) -> AudioSegment:
        """
        Divide o texto em segmentos de controle (VAD), gera o áudio e reconstrói com a cadência ideal.
        """
        # Divide por parágrafos para simular os "segmentos longos" do VAD
        paragrafos = [p.strip() for p in texto_completo.split('\n') if p.strip()]

        audio_final = AudioSegment.empty()

        print(f"Processando {len(paragrafos)} segmentos de texto...")

        for idx, parag in enumerate(paragrafos):
            # Garante que o segmento não estoure o limite estipulado na sua config
            # (Aproximação: 1 caractere consome aprox 0.07s de áudio)
            max_chars = int(self.config.optimize_max_segment_duration_s / 0.07)
            if len(parag) > max_chars:
                # Subdivide fragmentos longos baseando-se no limite do VAD
                sub_segmentos = re.split(r'(?<=[.!?])\s+', parag)
            else:
                sub_segmentos = [parag]

            for seg in sub_segmentos:
                if not seg.strip(): continue

                # Gera o áudio direto na memória via streaming de bytes
                communicate = edge_tts.Communicate(seg, self.config.voice_name)
                audio_bytes = b""
                async for chunk in communicate.stream():
                    if chunk["type"] == "audio":
                        audio_bytes += chunk["data"]

                if audio_bytes:
                    # Carrega os bytes na biblioteca de áudio
                    seg_audio = AudioSegment.from_file(io.BytesIO(audio_bytes), format="mp3")

                    # Aplica métricas de energia e silêncio
                    seg_processado = self.process_segment(seg_audio)

                    # Concatena aplicando o ganho de pausa configurado (VAD Fallback Min Pause)
                    audio_final += seg_processado + AudioSegment.silent(duration=self.config.pause_between_paragraphs_ms)

        return audio_final

# --- EXECUÇÃO DO PIPELINE ---

# 1. Carregar Configurações
config = AudioProcessorConfig()

# 2. Extrair Texto
pdf_path = "/content/v40n73a02.pdf"  # Certifique-se de que o arquivo está nesta pasta do Colab
texto_bruto = ""
reader = pypdf.PdfReader(pdf_path)

for page in reader.pages[:3]: # Primeiras 3 páginas para validação métrica
    # CORREÇÃO 3: 'page' já é o objeto da página, o acesso antigo quebrava o script
    text = page.extract_text()
    if text: texto_bruto += text + "\n"

texto_tratado = limpar_texto_pdf(texto_bruto)

# 3. Rodar Processamento de Sinais de Áudio
if texto_tratado:
    processor = AdvancedAudioProcessor(config)

    print("Iniciando pipeline de processamento acústico...")
    audio_gerado = await processor.text_to_speech_pipeline(texto_tratado)

    # Exportar resultado final
    arquivo_saida = "audio_pipeline_avancado.mp3"
    audio_gerado.export(arquivo_saida, format="mp3")

    print("\nPipeline finalizado com sucesso!")
    display(Audio(arquivo_saida, autoplay=False))
else:
    print("Erro ao extrair texto.")


In [ ]:
import pypdf
import asyncio
import edge_tts
import re
import io
import numpy as np
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
from IPython.display import Audio, display
from dataclasses import dataclass

# --- CONFIGURAÇÕES DO PIPELINE AVANÇADO ---
@dataclass
class AudioProcessorConfig:
    # Mapeamento de Personagens para Diarização
    voice_narrador: str = "pt-BR-FranciscaNeural"  # Voz do "Jornal"
    voice_entrevistado: str = "pt-BR-AntonioNeural" # Voz do "Fabio Herrmann"

    # Métricas de Silêncio e Energia
    enable_energy_trimming: bool = True
    trim_silence_duration_ms: int = 150

    # Ganho Estatístico e Normalização
    target_dbfs: float = -20.0  # Alvo de normalização (Volume constante)

    # Cadência e Respiração (VAD)
    optimize_max_segment_duration_s: int = 20
    pause_between_paragraphs_ms: int = 900  # Pausa entre falas
    pause_between_sentences_ms: int = 400   # Pausa menor para pontuações internas

# --- FILTRO E TRATAMENTO DE TEXTO AVANÇADO ---
def limpar_texto_pdf(texto_puro):
    if not texto_puro: return ""
    texto = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', texto_puro)
    texto = re.sub(r'(?<!\n)\n(?!\n)', ' ', texto)
    texto = re.sub(r'[ \t]+', ' ', texto)
    return texto.strip()

# --- PROCESSADOR DE ÁUDIO COM DIARIZAÇÃO E GANHO ESTATÍSTICO ---
class AdvancedAudioProcessor:
    def __init__(self, config: AudioProcessorConfig):
        self.config = config

    def aplicar_ganho_estatistico(self, audio_segment: AudioSegment) -> AudioSegment:
        """
        Calcula as propriedades estatísticas do áudio usando matrizes de dados
        e aplica ganho dinâmico baseado no dBFS alvo.
        """
        if not audio_segment or len(audio_segment) == 0:
            return audio_segment

        samples = np.array(audio_segment.get_array_of_samples())
        if np.all(samples == 0):
            return audio_segment

        # CORREÇÃO 1: Alterado de dbfs para dBFS (Case-sensitive)
        change_in_dbfs = self.config.target_dbfs - audio_segment.dBFS
        audio_normalizado = audio_segment.apply_gain(change_in_dbfs)

        # CORREÇÃO 2: Alterado de max_dbfs para max_dBFS (Case-sensitive)
        if audio_normalizado.max_dBFS > -1.0:
            audio_normalizado = audio_normalizado.apply_gain(-audio_normalizado.max_dBFS - 1.0)

        return audio_normalizado

    def process_segment(self, audio_segment: AudioSegment) -> AudioSegment:
        """Aplica trimming adaptativo baseado na energia média do sinal."""
        if not self.config.enable_energy_trimming or len(audio_segment) == 0:
            return audio_segment

        # Define o limiar dinamicamente baseado no dBFS atual do segmento
        threshold_dinamico = min(audio_segment.dBFS - 25, -45)

        non_silent_chunks = detect_nonsilent(
            audio_segment,
            min_silence_len=self.config.trim_silence_duration_ms,
            silence_thresh=int(threshold_dinamico)
        )

        if not non_silent_chunks:
            return audio_segment

        # Recupera os índices da matriz do detector de silêncio
        start_trim_ms = non_silent_chunks[0][0]
        end_trim_ms = non_silent_chunks[-1][1]

        start_ms = max(0, start_trim_ms - self.config.trim_silence_duration_ms)
        end_ms = min(len(audio_segment), end_trim_ms + self.config.trim_silence_duration_ms)

        return audio_segment[start_ms:end_ms]

    async def text_to_speech_pipeline(self, texto_completo: str) -> AudioSegment:
        """Analisa o texto linha por linha para chavear a voz dos falantes (Diarização)."""
        linhas = [l.strip() for l in texto_completo.split('\n') if l.strip()]
        audio_final = AudioSegment.empty()
        voz_atual = self.config.voice_narrador

        print(f"Processando {len(linhas)} blocos de texto estruturado...")

        for linha in linhas:
            # --- MECANISMO DE DIARIZAÇÃO ---
            if re.match(r'^(Fabio Herrmann\s*:)', linha, re.IGNORECASE):
                voz_atual = self.config.voice_entrevistado
                linha_limpa = re.sub(r'^(Fabio Herrmann\s*:)\s*', '', linha, flags=re.IGNORECASE)
                print(f"[Voz: Fabio Herrmann] -> {linha_limpa[:40]}...")
            elif re.match(r'^(Jornal\s*:)', linha, re.IGNORECASE):
                voz_atual = self.config.voice_narrador
                linha_limpa = re.sub(r'^(Jornal\s*:)\s*', '', linha, flags=re.IGNORECASE)
                print(f"[Voz: Jornal] -> {linha_limpa[:40]}...")
            else:
                linha_limpa = linha

            if not linha_limpa.strip():
                continue

            sub_segmentos = re.split(r'(?<=[.!?])\s+', linha_limpa)

            for seg in sub_segmentos:
                if not seg.strip(): continue

                communicate = edge_tts.Communicate(seg, voz_atual)
                audio_bytes = b""
                async for chunk in communicate.stream():
                    if chunk["type"] == "audio":
                        audio_bytes += chunk["data"]

                if audio_bytes:
                    seg_audio = AudioSegment.from_file(io.BytesIO(audio_bytes), format="mp3")

                    # Filtros Acústicos do Pipeline
                    seg_processado = self.process_segment(seg_audio)
                    seg_estatistico = self.aplicar_ganho_estatistico(seg_processado)

                    audio_final += seg_estatistico + AudioSegment.silent(duration=self.config.pause_between_sentences_ms)

            audio_final += AudioSegment.silent(duration=self.config.pause_between_paragraphs_ms)

        return audio_final

# --- EXECUÇÃO ---
config = AudioProcessorConfig()
pdf_path = "/content/v40n73a02.pdf"
texto_bruto = ""
reader = pypdf.PdfReader(pdf_path)

# Extração de texto limitada às primeiras páginas para teste dinâmico veloz
for page in reader.pages[:4]:
    text = page.extract_text()
    if text:
        texto_bruto += text + "\n"

if texto_bruto:
    processor = AdvancedAudioProcessor(config)
    print("Iniciando pipeline de Diarização Acústica...")

    audio_gerado = await processor.text_to_speech_pipeline(texto_bruto)

    arquivo_saida = "entrevista_diarizada.mp3"
    audio_gerado.export(arquivo_saida, format="mp3")

    print("\nProcessamento Concluído!")
    display(Audio(arquivo_saida, autoplay=False))
else:
    print("Erro ao extrair texto do PDF.")


In [ ]:
import pypdf
import asyncio
import edge_tts
import re
import io
import numpy as np
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
from IPython.display import Audio, display
from dataclasses import dataclass

# --- CONFIGURAÇÕES DO PIPELINE AVANÇADO ---
@dataclass
class AudioProcessorConfig:
    # Mapeamento de Personagens para Diarização
    voice_narrador: str = "pt-BR-FranciscaNeural"  # Voz do "Jornal"
    voice_entrevistado: str = "pt-BR-AntonioNeural" # Voz do "Fabio Herrmann"

    # Métricas de Silêncio e Energia
    enable_energy_trimming: bool = True
    trim_silence_duration_ms: int = 150

    # Ganho Estatístico e Normalização
    target_dbfs: float = -20.0  # Alvo de normalização (Volume constante)

    # Cadência e Respiração (VAD)
    optimize_max_segment_duration_s: int = 20
    pause_between_paragraphs_ms: int = 900  # Pausa entre falas
    pause_between_sentences_ms: int = 400   # Pausa menor para pontuações internas

# --- PROCESSADOR DE ÁUDIO COM DIARIZAÇÃO E GANHO ESTATÍSTICO ---
class AdvancedAudioProcessor:
    def __init__(self, config: AudioProcessorConfig):
        self.config = config

    def limpar_texto_pdf(self, texto_puro):
        """
        Trata quebras de linha de colunas e hifens órfãos ANTES da conversão.
        """
        if not texto_puro:
            return ""

        # 1. Junta palavras separadas por hífen no final da coluna/linha (ex: com- \n putador -> computador)
        texto = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', texto_puro)

        # 2. Corrige quebras de linha abruptas da coluna, mas PRESERVA quando a linha seguinte começa com os nomes dos falantes
        linhas_puras = texto.split('\n')
        linhas_tratadas = []

        for linha in linhas_puras:
            linha_st = linha.strip()
            if not linha_st:
                continue

            # Se a linha atual indica mudança de falante, ela DEVE começar em uma nova linha no texto final
            if re.match(r'^(Fabio Herrmann\s*:|Jornal\s*:)', linha_st, re.IGNORECASE):
                linhas_tratadas.append("\n" + linha_st)
            else:
                # Caso contrário, apenas junta com espaço (removendo a quebra artificial da coluna do PDF)
                linhas_tratadas.append(linha_st)

        # Reconstrói o texto e limpa múltiplos espaços em branco
        texto_final = " ".join(linhas_tratadas)
        texto_final = re.sub(r'[ \t]+', ' ', texto_final)

        return texto_final.strip()

    def aplicar_ganho_estatistico(self, audio_segment: AudioSegment) -> AudioSegment:
        if not audio_segment or len(audio_segment) == 0:
            return audio_segment

        samples = np.array(audio_segment.get_array_of_samples())
        if np.all(samples == 0):
            return audio_segment

        change_in_dbfs = self.config.target_dbfs - audio_segment.dBFS
        audio_normalizado = audio_segment.apply_gain(change_in_dbfs)

        if audio_normalizado.max_dBFS > -1.0:
            audio_normalizado = audio_normalizado.apply_gain(-audio_normalizado.max_dBFS - 1.0)

        return audio_normalizado

    def process_segment(self, audio_segment: AudioSegment) -> AudioSegment:
        if not self.config.enable_energy_trimming or len(audio_segment) == 0:
            return audio_segment

        threshold_dinamico = min(audio_segment.dBFS - 25, -45)

        non_silent_chunks = detect_nonsilent(
            audio_segment,
            min_silence_len=self.config.trim_silence_duration_ms,
            silence_thresh=int(threshold_dinamico)
        )

        if not non_silent_chunks:
            return audio_segment

        start_trim_ms = non_silent_chunks[0][0]
        end_trim_ms = non_silent_chunks[-1][1]

        start_ms = max(0, start_trim_ms - self.config.trim_silence_duration_ms)
        end_ms = min(len(audio_segment), end_trim_ms + self.config.trim_silence_duration_ms)

        return audio_segment[start_ms:end_ms]

    async def text_to_speech_pipeline(self, texto_bruto: str) -> AudioSegment:
        """Aplica a limpeza profunda e chaveia a voz dos falantes por linha."""
        # CORREÇÃO CRUCIAL: O texto é limpo aqui antes de qualquer processamento de áudio
        texto_limpo = self.limpar_texto_pdf(texto_bruto)

        # Divide o texto reconstruído de volta em blocos baseados nos parágrafos reais criados na limpeza
        linhas = [l.strip() for l in texto_limpo.split('\n') if l.strip()]
        audio_final = AudioSegment.empty()
        voz_atual = self.config.voice_narrador

        print(f"Processando {len(linhas)} blocos de texto estruturado e limpo...")

        for linha in linhas:
            # --- MECANISMO DE DIARIZAÇÃO ---
            if re.match(r'^(Fabio Herrmann\s*:)', linha, re.IGNORECASE):
                voz_atual = self.config.voice_entrevistado
                linha_limpa = re.sub(r'^(Fabio Herrmann\s*:)\s*', '', linha, flags=re.IGNORECASE)
                print(f"[Voz: Fabio Herrmann] -> {linha_limpa[:50]}...")
            elif re.match(r'^(Jornal\s*:)', linha, re.IGNORECASE):
                voz_atual = self.config.voice_narrador
                linha_limpa = re.sub(r'^(Jornal\s*:)\s*', '', linha, flags=re.IGNORECASE)
                print(f"[Voz: Jornal] -> {linha_limpa[:50]}...")
            else:
                linha_limpa = linha

            if not linha_limpa.strip():
                continue

            # Divide os períodos longos respeitando pontuações para o fôlego da IA
            sub_segmentos = re.split(r'(?<=[.!?])\s+', linha_limpa)

            for seg in sub_segmentos:
                if not seg.strip(): continue

                communicate = edge_tts.Communicate(seg, voz_atual)
                audio_bytes = b""
                async for chunk in communicate.stream():
                    if chunk["type"] == "audio":
                        audio_bytes += chunk["data"]

                if audio_bytes:
                    seg_audio = AudioSegment.from_file(io.BytesIO(audio_bytes), format="mp3")

                    # Filtros Acústicos do Pipeline
                    seg_processado = self.process_segment(seg_audio)
                    seg_estatistico = self.aplicar_ganho_estatistico(seg_processado)

                    audio_final += seg_estatistico + AudioSegment.silent(duration=self.config.pause_between_sentences_ms)

            audio_final += AudioSegment.silent(duration=self.config.pause_between_paragraphs_ms)

        return audio_final

# --- EXECUÇÃO DO PIPELINE ---
config = AudioProcessorConfig()
pdf_path = "/content/v40n73a02.pdf"
texto_bruto = ""
reader = pypdf.PdfReader(pdf_path)

# Extração do texto bruto direto do arquivo para as primeiras 4 páginas
for page in reader.pages[:4]:
    text = page.extract_text()
    if text:
        texto_bruto += text + "\n"

if texto_bruto:
    processor = AdvancedAudioProcessor(config)
    print("Iniciando pipeline de Diarização Acústica...")

    # Passamos o texto_bruto direto; o método interno cuida da higienização completa
    audio_gerado = await processor.text_to_speech_pipeline(texto_bruto)

    arquivo_saida = "entrevista_diarizada.mp3"
    audio_gerado.export(arquivo_saida, format="mp3")

    print("\nProcessamento Concluído com Sucesso!")
    display(Audio(arquivo_saida, autoplay=False))
else:
    print("Erro ao extrair texto do PDF.")


In [ ]:
import pypdf
import asyncio
import edge_tts
import re
import io
import numpy as np
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
from IPython.display import Audio, display
from dataclasses import dataclass

# --- CONFIGURAÇÕES DO PIPELINE AVANÇADO ---
@dataclass
class AudioProcessorConfig:
    # Mapeamento de Personagens para Diarização
    voice_narrador: str = "pt-BR-FranciscaNeural"  # Voz do "Jornal"
    voice_entrevistado: str = "pt-BR-AntonioNeural" # Voz do "Fabio Herrmann"

    # Métricas de Silêncio e Energia
    enable_energy_trimming: bool = True
    trim_silence_duration_ms: int = 150

    # Ganho Estatístico e Normalização
    target_dbfs: float = -20.0  # Alvo de volume constante

    # Cadência e Respiração (VAD)
    optimize_max_segment_duration_s: int = 20
    pause_between_paragraphs_ms: int = 1100  # Pausa um pouco maior na troca de falante
    pause_between_sentences_ms: int = 400   # Pausa menor interna

# --- PROCESSADOR DE ÁUDIO COM DIARIZAÇÃO E REMOÇÃO DE RUÍDOS DE PDF ---
class AdvancedAudioProcessor:
    def __init__(self, config: AudioProcessorConfig):
        self.config = config

    def limpar_texto_pdf(self, texto_puro):
        """
        Remove notas de rodapé, cabeçalhos e limpa hifens órfãos de colunas,
        garantindo o isolamento dos marcadores de fala.
        """
        if not texto_puro:
            return ""

        # --- 1. REMOÇÃO DE CABEÇALHOS E RODAPÉS (REGEX) ---
        # Remove a nota de rodapé dinâmica do Jornal de Psicanálise (inclui variação de páginas)
        texto = re.sub(r'Jornal\s+de\s+Psicanálise,\s*São\s*Paulo,?\s*\d+\(\d+\):\s*\d+-\d+,\s*\w+\.?\s*\d+\.?\s*\d*', '', texto_puro, flags=re.IGNORECASE)
        # Remove o cabeçalho repetitivo das páginas parciais
        texto = re.sub(r'Entrevista\s+com\s+Fabio\s+Herrmann\s+em\s+2000', '', texto, flags=re.IGNORECASE)

        # --- 2. TRATAMENTO DE HIFENS DE COLUNA ---
        # Junta palavras separadas por hífen no final de uma linha física
        texto = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', texto)

        # --- 3. RECONSTRUÇÃO INTELIGENTE DE DIÁLOGOS ---
        linhas_puras = texto.split('\n')
        linhas_tratadas = []
        bloco_atual = ""

        for linha in linhas_puras:
            linha_st = linha.strip()
            if not linha_st:
                continue

            # Detecta se a linha atual inicia uma nova fala (Jornal: ou Fabio Herrmann:)
            is_jornal = re.match(r'^(Jornal\s*:)', linha_st, re.IGNORECASE)
            is_fabio = re.match(r'^(Fabio\s+Herrmann\s*:|Fabio\s*:)', linha_st, re.IGNORECASE)

            if is_jornal or is_fabio:
                # Se já havia um bloco sendo construído, salva-o antes de abrir a nova fala
                if bloco_atual.strip():
                    linhas_tratadas.append(bloco_atual.strip())
                # Inicia o novo bloco contendo o marcador original intacto
                bloco_atual = linha_st
            else:
                # Se não for início de diálogo, é continuação do parágrafo da coluna anterior.
                # Concatena com um espaço simples.
                if bloco_atual:
                    bloco_atual += " " + linha_st
                else:
                    bloco_atual = linha_st

        # Adiciona o último bloco processado
        if bloco_atual.strip():
            linhas_tratadas.append(bloco_atual.strip())

        # Junta tudo usando quebras de linha explícitas (\n) para isolar os falantes
        texto_final = "\n".join(linhas_tratadas)
        # Remove espaços duplos remanescentes
        texto_final = re.sub(r'[ \t]+', ' ', texto_final)

        return texto_final.strip()

    def aplicar_ganho_estatistico(self, audio_segment: AudioSegment) -> AudioSegment:
        if not audio_segment or len(audio_segment) == 0:
            return audio_segment

        samples = np.array(audio_segment.get_array_of_samples())
        if np.all(samples == 0):
            return audio_segment

        change_in_dbfs = self.config.target_dbfs - audio_segment.dBFS
        audio_normalizado = audio_segment.apply_gain(change_in_dbfs)

        if audio_normalizado.max_dBFS > -1.0:
            audio_normalizado = audio_normalizado.apply_gain(-audio_normalizado.max_dBFS - 1.0)

        return audio_normalizado

    def process_segment(self, audio_segment: AudioSegment) -> AudioSegment:
        if not self.config.enable_energy_trimming or len(audio_segment) == 0:
            return audio_segment

        threshold_dinamico = min(audio_segment.dBFS - 25, -45)

        non_silent_chunks = detect_nonsilent(
            audio_segment,
            min_silence_len=self.config.trim_silence_duration_ms,
            silence_thresh=int(threshold_dinamico)
        )

        if not non_silent_chunks:
            return audio_segment

        start_trim_ms = non_silent_chunks[0][0]
        end_trim_ms = non_silent_chunks[-1][1]

        start_ms = max(0, start_trim_ms - self.config.trim_silence_duration_ms)
        end_ms = min(len(audio_segment), end_trim_ms + self.config.trim_silence_duration_ms)

        return audio_segment[start_ms:end_ms]

    async def text_to_speech_pipeline(self, texto_bruto: str) -> AudioSegment:
        """Limpa metadados e processa o chaveamento exato das vozes por bloco de diálogo."""
        texto_limpo = self.limpar_texto_pdf(texto_bruto)

        # Cada item na lista agora representa rigorosamente uma fala inteira isolada
        linhas = [l.strip() for l in texto_limpo.split('\n') if l.strip()]
        audio_final = AudioSegment.empty()
        voz_atual = self.config.voice_narrador

        print(f"Processando {len(linhas)} blocos de diálogos estruturados e limpos...")

        for linha in linhas:
            # --- MECANISMO DE DIARIZAÇÃO REFINADO ---
            if re.match(r'^(Fabio\s+Herrmann\s*:|Fabio\s*:)', linha, re.IGNORECASE):
                voz_atual = self.config.voice_entrevistado
                linha_limpa = re.sub(r'^(Fabio\s+Herrmann\s*:|Fabio\s*:)\s*', '', linha, flags=re.IGNORECASE)
                print(f"[Voz MASCULINA: Fabio H.] -> {linha_limpa[:60]}...")
            elif re.match(r'^(Jornal\s*:)', linha, re.IGNORECASE):
                voz_atual = self.config.voice_narrador
                linha_limpa = re.sub(r'^(Jornal\s*:)\s*', '', linha, flags=re.IGNORECASE)
                print(f"[Voz FEMININA: Jornal] -> {linha_limpa[:60]}...")
            else:
                # Se por acaso sobrar texto sem rótulo direto, mantém a voz anterior (continuação)
                linha_limpa = linha

            if not linha_limpa.strip():
                continue

            # Divide frases longas internamente para a respiração natural da IA
            sub_segmentos = re.split(r'(?<=[.!?])\s+', linha_limpa)

            for seg in sub_segmentos:
                if not seg.strip(): continue

                communicate = edge_tts.Communicate(seg, voz_atual)
                audio_bytes = b""
                async for chunk in communicate.stream():
                    if chunk["type"] == "audio":
                        audio_bytes += chunk["data"]

                if audio_bytes:
                    seg_audio = AudioSegment.from_file(io.BytesIO(audio_bytes), format="mp3")

                    # Processamento de Sinais Acústicos
                    seg_processado = self.process_segment(seg_audio)
                    seg_estatistico = self.aplicar_ganho_estatistico(seg_processado)

                    audio_final += seg_estatistico + AudioSegment.silent(duration=self.config.pause_between_sentences_ms)

            # Adiciona a pausa de troca de turno/parágrafo
            audio_final += AudioSegment.silent(duration=self.config.pause_between_paragraphs_ms)

        return audio_final

# --- EXECUÇÃO DO PIPELINE ---
config = AudioProcessorConfig()
pdf_path = "/content/v40n73a02.pdf"
texto_bruto = ""
reader = pypdf.PdfReader(pdf_path)

# Extração inicial de teste (páginas 1 a 4)
for page in reader.pages[:4]:
    text = page.extract_text()
    if text:
        texto_bruto += text + "\n"

if texto_bruto:
    processor = AdvancedAudioProcessor(config)
    print("Iniciando pipeline de Diarização Acústica...")

    audio_gerado = await processor.text_to_speech_pipeline(texto_bruto)

    arquivo_saida = "entrevista_diarizada_final.mp3"
    audio_gerado.export(arquivo_saida, format="mp3")

    print("\nProcessamento Concluído!")
    display(Audio(arquivo_saida, autoplay=False))
else:
    print("Erro ao extrair texto do PDF.")


In [ ]:
!apt-get install -y ffmpeg
!pip install pypdf edge-tts pydub tqdm numpy

In [ ]:
import pypdf
import asyncio
import edge_tts
import re
import io
import numpy as np
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
from IPython.display import Audio, display
from dataclasses import dataclass
from tqdm.notebook import tqdm

# --- CONFIGURAÇÕES DO PIPELINE AVANÇADO ---
@dataclass
class AudioProcessorConfig:
    voice_narrador: str = "pt-BR-FranciscaNeural"   # JORNAL / INTRODUÇÃO
    voice_entrevistado: str = "pt-BR-AntonioNeural"  # FABIO

    enable_energy_trimming: bool = True
    trim_silence_duration_ms: int = 150
    target_dbfs: float = -20.0

    pause_between_paragraphs_ms: int = 1500  # Pausa ligeiramente maior entre turnos de fala

class AdvancedAudioProcessor:
    def __init__(self, config: AudioProcessorConfig):
        self.config = config

    def limpar_texto_pdf(self, texto_puro):
        """
        Padroniza os marcadores de fala para JORNAL: e FABIO:
        e limpa ruídos repetitivos de cabeçalho/rodapé.
        """
        if not texto_puro:
            return ""

        # 1. Remoção de rodapés e cabeçalhos recorrentes
        texto = re.sub(r'Jornal\s+de\s+Psicanálise,\s*São\s*Paulo,?\s*\d+\(\d+\):\s*\d+-\d+,\s*\w+\.?\s*\d+\.?\s*\d*', '', texto_puro, flags=re.IGNORECASE)
        texto = re.sub(r'Entrevista\s+com\s+Fabio\s+Herrmann\s+em\s+2000', '', texto, flags=re.IGNORECASE)

        # 2. Junta hifens de quebra de linha/coluna
        texto = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', texto)

        # 3. Normalização estrita dos falantes
        texto = re.sub(r'J\s*o\s*r\s*n\s*a\s*l\s*:', '\nJORNAL:\n', texto, flags=re.IGNORECASE)
        texto = re.sub(r'F\s*a\s*b\s*i\s*o\s*(?:\s*H\s*e\s*r\s*r\s*m\s*a\s*n\s*n\s*)?:', '\nFABIO:\n', texto, flags=re.IGNORECASE)

        linhas_puras = texto.split('\n')
        linhas_tratadas = []
        bloco_atual = ""
        # Inicializa como JORNAL: para garantir que o texto introdutório da Leda Herrmann seja narrado
        falante_atual = "JORNAL:"

        for linha in linhas_puras:
            linha_st = linha.strip()
            if not linha_st:
                continue

            if linha_st == "JORNAL:":
                if bloco_atual.strip():
                    linhas_tratadas.append(f"{falante_atual} {bloco_atual.strip()}")
                falante_atual = "JORNAL:"
                bloco_atual = ""
            elif linha_st == "FABIO:":
                if bloco_atual.strip():
                    linhas_tratadas.append(f"{falante_atual} {bloco_atual.strip()}")
                falante_atual = "FABIO:"
                bloco_atual = ""
            else:
                bloco_atual += " " + linha_st

        if bloco_atual.strip():
            linhas_tratadas.append(f"{falante_atual} {bloco_atual.strip()}")

        texto_final = "\n".join(linhas_tratadas)
        texto_final = re.sub(r'[ \t]+', ' ', texto_final)

        return texto_final.strip()

    def aplicar_ganho_estatistico(self, audio_segment: AudioSegment) -> AudioSegment:
        if not audio_segment or len(audio_segment) == 0: return audio_segment
        samples = np.array(audio_segment.get_array_of_samples())
        if np.all(samples == 0): return audio_segment

        change_in_dbfs = self.config.target_dbfs - audio_segment.dBFS
        audio_normalizado = audio_segment.apply_gain(change_in_dbfs)

        if audio_normalizado.max_dBFS > -1.0:
            audio_normalizado = audio_normalizado.apply_gain(-audio_normalizado.max_dBFS - 1.0)
        return audio_normalizado

    def process_segment(self, audio_segment: AudioSegment) -> AudioSegment:
        if not self.config.enable_energy_trimming or len(audio_segment) == 0: return audio_segment
        threshold_dinamico = min(audio_segment.dBFS - 25, -45)

        non_silent_chunks = detect_nonsilent(
            audio_segment,
            min_silence_len=self.config.trim_silence_duration_ms,
            silence_thresh=int(threshold_dinamico)
        )
        if not non_silent_chunks: return audio_segment

        # CORREÇÃO AQUI: Acessando os índices corretos da lista de tuplas [(start, end)]
        start_trim_ms = non_silent_chunks[0][0]
        end_trim_ms = non_silent_chunks[-1][1]

        start_ms = max(0, start_trim_ms - self.config.trim_silence_duration_ms)
        end_ms = min(len(audio_segment), end_trim_ms + self.config.trim_silence_duration_ms)
        return audio_segment[start_ms:end_ms]

    async def text_to_speech_pipeline(self, texto_bruto: str) -> AudioSegment:
        texto_limpo = self.limpar_texto_pdf(texto_bruto)
        linhas = [l.strip() for l in texto_limpo.split('\n') if l.strip()]

        total_blocos = len(linhas)
        print(f"\n==============================================")
        print(f"📊 MÉTRICAS DE DIARIZAÇÃO GLOBAL:")
        print(f"-> Total de turnos de fala mapeados: {total_blocos}")

        jornal_count = sum(1 for l in linhas if l.startswith("JORNAL:"))
        fabio_count = sum(1 for l in linhas if l.startswith("FABIO:"))
        print(f"-> Total Blocos JORNAL (Perguntas/Intro): {jornal_count}")
        print(f"-> Total Blocos FABIO (Respostas): {fabio_count}")
        print(f"==============================================\n")

        audio_final = AudioSegment.empty()

        print("Iniciando geração sequencial por turnos de fala...")

        for idx, linha in enumerate(tqdm(linhas, desc="Convertendo Diálogos", unit="bloco")):
            if linha.startswith("FABIO:"):
                voz_atual = self.config.voice_entrevistado
                linha_limpa = linha.replace("FABIO:", "").strip()
            elif linha.startswith("JORNAL:"):
                voz_atual = self.config.voice_narrador
                linha_limpa = linha.replace("JORNAL:", "").strip()
            else:
                voz_atual = self.config.voice_narrador
                linha_limpa = linha

            if not linha_limpa.strip():
                continue

            # CORREÇÃO AQUI: Enviamos o bloco completo (ou parágrafo) em vez de frase por frase.
            # Isso evita o bloqueio de IP por excesso de requisições HTTP e acelera o processo.
            try:
                communicate = edge_tts.Communicate(linha_limpa, voz_atual)
                audio_bytes = b""
                async for chunk in communicate.stream():
                    if chunk["type"] == "audio":
                        audio_bytes += chunk["data"]

                if audio_bytes:
                    seg_audio = AudioSegment.from_file(io.BytesIO(audio_bytes), format="mp3")
                    seg_processado = self.process_segment(seg_audio)
                    seg_estatistico = self.aplicar_gain_estatistico(seg_processado) if hasattr(self, 'aplicar_gain_estatistico') else self.aplicar_ganho_estatistico(seg_processado)

                    # Adiciona o áudio gerado e a pausa de transição de fala
                    audio_final += seg_estatistico + AudioSegment.silent(duration=self.config.pause_between_paragraphs_ms)
            except Exception as e:
                # Agora o erro é impresso para sabermos se algum bloco falhou
                print(f"⚠️ Erro ao processar o bloco {idx}: {e}")

        return audio_final

# --- EXECUÇÃO DO PIPELINE COMPLETO ---
config = AudioProcessorConfig()
pdf_path = "/content/v40n73a02.pdf"
texto_bruto = ""

try:
    reader = pypdf.PdfReader(pdf_path)
    print(f"Extraindo texto de todas as {len(reader.pages)} páginas do documento...")
    for page in tqdm(reader.pages, desc="Lendo Páginas do PDF", unit="pág"):
        text = page.extract_text()
        if text:
            texto_bruto += text + "\n"
except Exception as e:
    print(f"Erro ao abrir ou ler o PDF: {e}")

if texto_bruto:
    processor = AdvancedAudioProcessor(config)
    print("\nIniciando processamento acústico (TTS)...")

    # Executa a função assíncrona diretamente no ecossistema do Colab
    audio_gerado = await processor.text_to_speech_pipeline(texto_bruto)

    arquivo_saida = "entrevista_completa_diarizada.mp3"
    print("\n💾 Compilando e exportando o arquivo final de áudio...")
    audio_gerado.export(arquivo_saida, format="mp3")

    print("\n✨ Pipeline Finalizado com Sucesso!")
    display(Audio(arquivo_saida, autoplay=False))
else:
    print("Erro: Nenhum texto pôde ser extraído do arquivo PDF.")

In [ ]:
import pypdf
import asyncio
import edge_tts
import re
import io
import numpy as np
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
from IPython.display import Audio, display
from dataclasses import dataclass
from tqdm.notebook import tqdm

# --- CONFIGURAÇÕES DO PIPELINE AVANÇADO ---
@dataclass
class AudioProcessorConfig:
    voice_narrador: str = "pt-BR-FranciscaNeural"   # JORNAL / INTRODUÇÃO
    voice_entrevistado: str = "pt-BR-AntonioNeural"  # FABIO

    enable_energy_trimming: bool = True
    trim_silence_duration_ms: int = 150
    target_dbfs: float = -20.0

    pause_between_paragraphs_ms: int = 1500  # Pausa confortável entre turnos de fala

class AdvancedAudioProcessor:
    def __init__(self, config: AudioProcessorConfig):
        self.config = config

    def limpar_texto_pdf(self, texto_puro):
        """
        Aplica remoção agressiva de rodapés e cabeçalhos usando o coringa [^\n]*
        e padroniza os marcadores de fala.
        """
        if not texto_puro:
            return ""

        # 1. REMOÇÃO AGRESSIVA DE RUÍDOS (Deleta a linha inteira a partir do termo)
        texto = re.sub(r'Jornal\s+de\s+Psicanálise[^\n]*', '', texto_puro, flags=re.IGNORECASE)
        texto = re.sub(r'Entrevista\s+com\s+Fabio\s+Herrmann\s+em\s+2000[^\n]*', '', texto, flags=re.IGNORECASE)

        # Remove números de página que ficaram isolados em linhas próprias
        texto = re.sub(r'^\s*\d+\s*$', '', texto, flags=re.MULTILINE)

        # 2. Junta hifens de quebra de linha/coluna
        texto = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', texto)

        # 3. NORMALIZAÇÃO ESTREITA DOS FALANTES
        texto = re.sub(r'J\s*o\s*r\s*n\s*a\s*l\s*:', '\nJORNAL:\n', texto, flags=re.IGNORECASE)
        texto = re.sub(r'F\s*a\s*b\s*i\s*o\s*(?:\s*H\s*e\s*r\s*r\s*m\s*a\s*n\s*n\s*)?:', '\nFABIO:\n', texto, flags=re.IGNORECASE)

        linhas_puras = texto.split('\n')
        linhas_tratadas = []
        bloco_atual = ""
        falante_atual = "JORNAL:"  # Inicializado como JORNAL: para garantir a narração da introdução editorial

        for linha in linhas_puras:
            linha_st = linha.strip()
            if not linha_st:
                continue

            if linha_st == "JORNAL:":
                if bloco_atual.strip():
                    linhas_tratadas.append(f"{falante_atual} {bloco_atual.strip()}")
                falante_atual = "JORNAL:"
                bloco_atual = ""
            elif linha_st == "FABIO:":
                if bloco_atual.strip():
                    linhas_tratadas.append(f"{falante_atual} {bloco_atual.strip()}")
                falante_atual = "FABIO:"
                bloco_atual = ""
            else:
                bloco_atual += " " + linha_st

        if bloco_atual.strip():
            linhas_tratadas.append(f"{falante_atual} {bloco_atual.strip()}")

        texto_final = "\n".join(linhas_tratadas)
        texto_final = re.sub(r'[ \t]+', ' ', texto_final)

        return texto_final.strip()

    def aplicar_ganho_estatistico(self, audio_segment: AudioSegment) -> AudioSegment:
        if not audio_segment or len(audio_segment) == 0: return audio_segment
        samples = np.array(audio_segment.get_array_of_samples())
        if np.all(samples == 0): return audio_segment

        change_in_dbfs = self.config.target_dbfs - audio_segment.dBFS
        audio_normalizado = audio_segment.apply_gain(change_in_dbfs)

        if audio_normalizado.max_dBFS > -1.0:
            audio_normalizado = audio_normalizado.apply_gain(-audio_normalizado.max_dBFS - 1.0)
        return audio_normalizado

    def process_segment(self, audio_segment: AudioSegment) -> AudioSegment:
        if not self.config.enable_energy_trimming or len(audio_segment) == 0: return audio_segment
        threshold_dinamico = min(audio_segment.dBFS - 25, -45)

        non_silent_chunks = detect_nonsilent(
            audio_segment,
            min_silence_len=self.config.trim_silence_duration_ms,
            silence_thresh=int(threshold_dinamico)
        )
        if not non_silent_chunks: return audio_segment

        # Acessando corretamente as tuplas do pydub [(start, end)]
        start_trim_ms = non_silent_chunks[0][0]
        end_trim_ms = non_silent_chunks[-1][1]

        start_ms = max(0, start_trim_ms - self.config.trim_silence_duration_ms)
        end_ms = min(len(audio_segment), end_trim_ms + self.config.trim_silence_duration_ms)
        return audio_segment[start_ms:end_ms]

    async def text_to_speech_pipeline(self, texto_bruto: str) -> AudioSegment:
        texto_limpo = self.limpar_texto_pdf(texto_bruto)
        linhas = [l.strip() for l in texto_limpo.split('\n') if l.strip()]

        total_blocos = len(linhas)
        print(f"\n==============================================")
        print(f"📊 MÉTRICAS DE DIARIZAÇÃO GLOBAL (27 PÁGINAS):")
        print(f"-> Total de turnos de fala mapeados: {total_blocos}")

        jornal_count = sum(1 for l in linhas if l.startswith("JORNAL:"))
        fabio_count = sum(1 for l in linhas if l.startswith("FABIO:"))
        print(f"-> Total Blocos JORNAL (Perguntas/Intro): {jornal_count}")
        print(f"-> Total Blocos FABIO (Respostas): {fabio_count}")
        print(f"==============================================\n")

        audio_final = AudioSegment.empty()

        print("Iniciando geração sequencial por blocos integrados de diálogo...")

        for idx, linha in enumerate(tqdm(linhas, desc="Convertendo Diálogos", unit="bloco")):
            if linha.startswith("FABIO:"):
                voz_atual = self.config.voice_entrevistado
                linha_limpa = linha.replace("FABIO:", "").strip()
            elif linha.startswith("JORNAL:"):
                voz_atual = self.config.voice_narrador
                linha_limpa = linha.replace("JORNAL:", "").strip()
            else:
                voz_atual = self.config.voice_narrador
                linha_limpa = linha

            if not linha_limpa.strip():
                continue

            try:
                # Enviando o bloco de fala completo para otimizar requisições HTTP e evitar Erro 429
                communicate = edge_tts.Communicate(linha_limpa, voz_atual)
                audio_bytes = b""
                async for chunk in communicate.stream():
                    if chunk["type"] == "audio":
                        audio_bytes += chunk["data"]

                if audio_bytes:
                    seg_audio = AudioSegment.from_file(io.BytesIO(audio_bytes), format="mp3")
                    seg_processado = self.process_segment(seg_audio)
                    seg_estatistico = self.aplicar_ganho_estatistico(seg_processado)

                    # Une os áudios inserindo o espaçamento configurado de parágrafo
                    audio_final += seg_estatistico + AudioSegment.silent(duration=self.config.pause_between_paragraphs_ms)
            except Exception as e:
                print(f"⚠️ Erro crítico no bloco {idx}: {e}")

        return audio_final

# --- EXECUÇÃO DO PIPELINE ---
config = AudioProcessorConfig()
pdf_path = "/content/v40n73a02.pdf"
texto_bruto = ""

try:
    reader = pypdf.PdfReader(pdf_path)
    print(f"Extraindo texto de todas as {len(reader.pages)} páginas do documento...")
    for page in tqdm(reader.pages, desc="Lendo Páginas do PDF", unit="pág"):
        text = page.extract_text()
        if text:
            texto_bruto += text + "\n"
except Exception as e:
    print(f"Erro ao abrir ou ler o PDF: {e}")

if texto_bruto:
    processor = AdvancedAudioProcessor(config)
    print("\nIniciando processamento acústico (TTS)...")

    # Execução assíncrona única e limpa no loop do Colab
    audio_gerado = await processor.text_to_speech_pipeline(texto_bruto)

    arquivo_saida = "entrevista_completa_diarizada.mp3"
    print("\n💾 Compilando e exportando o arquivo final de áudio...")
    audio_gerado.export(arquivo_saida, format="mp3")

    print("\n✨ Pipeline Finalizado com Sucesso Absoluto!")
    display(Audio(arquivo_saida, autoplay=False))
else:
    print("Erro: Nenhum texto extraído do PDF.")

### Visualizar Texto Limpo

Para inspecionar o texto depois da fase de limpeza e antes de ser processado para áudio, podemos chamar o método `limpar_texto_pdf` diretamente. Este método agora retorna o `texto_final` que inclui a formatação dos falantes.

In [ ]:
# Re-instanciar o processador para ter acesso ao método
processor = AdvancedAudioProcessor(config)

# Chamar o método de limpeza com o texto bruto original
texto_limpo_para_avaliacao = processor.limpar_texto_pdf(texto_bruto)

# Imprimir o texto limpo para avaliação
print(texto_limpo_para_avaliacao)

# Opcional: imprimir o número de linhas resultantes para ver a diarização
print(f"\nNúmero de linhas tratadas para diarização: {len(texto_limpo_para_avaliacao.split('\n'))}")

In [ ]:
teste_a = "/content/PDF_A_Digital.pdf"
teste_b = "/content/PDF_B_Digital.pdf"
teste_c = "/content/PDF_C_Digitalizado.pdf"

In [ ]:
import pypdf
from tqdm.notebook import tqdm

pdf_files = {
    "PDF_A_Digital": teste_a,
    "PDF_B_Digital": teste_b,
    "PDF_C_Digitalizado": teste_c
}

for name, path in pdf_files.items():
    print(f"\n{'='*50}")
    print(f"EXTRAINDO TEXTO DE: {name} ({path})")
    print(f"{'='*50}")

    extracted_text = ""
    try:
        reader = pypdf.PdfReader(path)
        print(f"Total de {len(reader.pages)} páginas.")
        for page_num, page in enumerate(tqdm(reader.pages, desc=f"Lendo Páginas de {name}", unit="pág")):
            text = page.extract_text()
            if text:
                extracted_text += f"--- Página {page_num + 1} ---\n" + text + "\n\n"
            else:
                extracted_text += f"--- Página {page_num + 1} (Vazia ou sem texto extraível) ---\n\n"

        print("\n--- TEXTO EXTRAÍDO ---")
        print(extracted_text)
    except Exception as e:
        print(f"Erro ao processar o arquivo {name}: {e}")

    print(f"\n{'='*50}\n")